# Inference Notebook — Fine-tuned TinyLlama from S3
This notebook is fully standalone — just run it on its own without needing to run the training

## 1. Install Requirements

In [1]:
!pip install -q transformers peft accelerate bitsandbytes boto3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.5 MB/s eta 0:00:00


## 2. Download Fine-tuned Model from S3

In [ ]:
import boto3
import os
from google.colab import userdata

# ── Get AWS Credentials safely 
AWS_SECRET_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')

# Check if keys exist
if not AWS_ACCESS_KEY or not AWS_SECRET_KEY:
    raise ValueError(" AWS credentials not found in Colab Secrets")

# Remove any accidental spaces
AWS_ACCESS_KEY = AWS_ACCESS_KEY.strip()
AWS_SECRET_KEY = AWS_SECRET_KEY.strip()

#  Config 
BUCKET    = 'cloud-project-time4'
S3_PREFIX = 'models/llama-finetuned/'
LOCAL_DIR = './inference-model'

os.makedirs(LOCAL_DIR, exist_ok=True)

#  Create S3 client correctly (no env issues) 
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

#  Try connection first (debug step)
try:
    s3.list_buckets()
    print("AWS connection successful")
except Exception as e:
    raise RuntimeError(f" AWS connection failed: {e}")

#  Download files 
downloaded = []

try:
    paginator = s3.get_paginator('list_objects_v2')

    for page in paginator.paginate(Bucket=BUCKET, Prefix=S3_PREFIX):
        for obj in page.get('Contents', []):
            key = obj['Key']
            filename = os.path.basename(key)

            if not filename:
                continue

            local_path = os.path.join(LOCAL_DIR, filename)

            s3.download_file(BUCKET, key, local_path)
            downloaded.append(filename)

            print(f'Downloaded: {filename}')

    print(f'\n {len(downloaded)} file(s) ready in {LOCAL_DIR}')
    print('Files:', os.listdir(LOCAL_DIR))

except Exception as e:
    print(f" Error during download: {e}")

AWS connection successful
Downloaded: README.md
Downloaded: adapter_config.json
Downloaded: adapter_model.safetensors
Downloaded: chat_template.jinja
Downloaded: tokenizer.json
Downloaded: tokenizer_config.json
Downloaded: training_args.bin

 7 file(s) ready in ./inference-model
Files: ['tokenizer.json', 'adapter_model.safetensors', 'README.md', 'chat_template.jinja', 'training_args.bin', 'adapter_config.json', 'tokenizer_config.json']


## 3. Load Base Model + LoRA Adapter

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Base model (4-bit — same quantization it was trained with)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4'
    ),
    device_map='auto'
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, LOCAL_DIR)
model.eval()
print('Fine-tuned model loaded and ready!')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine-tuned model loaded and ready!


## 4. Inference Function

In [11]:
def generate_answer(problem: str, max_new_tokens: int = 256, temperature: float = 0.3) -> str:
    """
    Takes a problem and returns the answer from the fine-tuned model
    """
    # Same template it was trained with
    prompt = f'<|user|>\n{problem}</s>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('generate_answer() ready!')


generate_answer() ready!


## 5. Test — Try the Model

In [12]:
test_problems = [
    'What is 25% of 200?',
    'If a train travels at 60 km/h for 2.5 hours, how far does it go?',
    'Solve for x: 3x + 7 = 22',
]

for problem in test_problems:
    print('=' * 60)
    print(f'Problem : {problem}')
    print(f'Answer  : {generate_answer(problem)}')
print('=' * 60)


Problem : What is 25% of 200?
Answer  : To find the value of 25% of 200, we need to find the value of 200 divided by 25.

200 divided by 25 is 80.

Therefore, the value of 25% of 200 is:
\[ 25 \cdot 80 = 200 \]

So, the answer is:
\[ \boxed{200} \]
Problem : If a train travels at 60 km/h for 2.5 hours, how far does it go?
Answer  : Let's break down the problem.

The train travels at 60 km/h for 2.5 hours.

The distance traveled in 2.5 hours is:
\[ 2.5 \times 60 = 150 \]

The distance traveled in 2 hours is:
\[ 2 \times 60 = 120 \]

The distance traveled in 1 hour is:
\[ 1 \times 60 = 60 \]

Thus, the distance traveled is:
\[ \boxed{120} \]
Problem : Solve for x: 3x + 7 = 22
Answer  : We have three variables: $x$, $y$, and $z$.

The equation $3x + 7 = 22$ can be rearranged as:
\[ 3x + 7 = 22 - 3x \]

So, we need to find the value of $x$ that satisfies this equation.

The equation $3x + 7 = 22 - 3x$ can be rearranged as:
\[ 3x + 7 = 22 - 3x - 3x \]

Substitute $x = 0$ into the equation:


## 6. Try Your Own Question

In [13]:
my_problem = 'Sara bought 3 pens, and the price of each pen is 5 pounds. Then she bought a notebook for 10 pounds. How much did Sara pay in total?'  # ← change this

answer = generate_answer(my_problem)
print(f' Problem : {my_problem}')
print(f' Answer  : {answer}')


 Problem : Sara bought 3 pens, and the price of each pen is 5 pounds. Then she bought a notebook for 10 pounds. How much did Sara pay in total?
 Answer  : Let's break down the problem step by step:

1. Sara bought 3 pens. Each pen costs 5 pounds.
2. The price of each pen is 5 pounds.
3. The total price is 3 * 5 = 15 pounds.
4. Sara bought a notebook for 10 pounds.
5. The price of a notebook is 10 pounds.
6. The total price is 15 + 10 = 25 pounds.
7. Sara paid 15 pounds for the pens and 25 pounds for the notebook.

So, Sara paid \boxed{15} pounds for the pens and \boxed{25} pounds for the notebook.
